# LLM Simulation Workbench

This notebook runs the LLM rewriting simulation pipeline: it loads datasets, applies personality/style rewrites, tests multiple providers, and exports outputs for factual consistency auditing.

## 1) Initial Setup

In this step, we import core libraries and load environment variables to validate API keys.

In [ ]:
from pathlib import Path
import os
import re
import requests

import pandas as pd
from dotenv import load_dotenv

In [ ]:
load_dotenv()

for env_var in ["GEMINI_API_KEY", "OPENROUTER_API_KEY", "DEEPSEEK_API_KEY"]:
    if os.getenv(env_var):
        print(f"{env_var} carregada do .env com sucesso.")
    else:
        print(f"{env_var} nao encontrada. Verifique o arquivo .env.")

## 2) Free Model Catalog (OpenRouter)

Here we query OpenRouter for available models and list only the ones that end with `:free`.

In [ ]:
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
	headers = {"Authorization": f"Bearer {api_key}"}

	resp = requests.get("https://openrouter.ai/api/v1/models", headers=headers, timeout=30)
	resp.raise_for_status()

	models = resp.json().get("data", [])
	free_models = sorted([m["id"] for m in models if ":free" in m.get("id", "")])

	print(f"Total free ativos: {len(free_models)}")
	for mid in free_models[:30]:
		print(mid)

## 3) Dataset Loading and Organization

In this section, we load files from the `data` folder, build DataFrames, and map names for easier experimentation.

In [ ]:
def read_dataset(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".json":
        return pd.read_json(file_path)

    raise ValueError(f"Formato de arquivo nao suportado: {file_path.name}")


data_dir = Path("../data")
dataset_files = sorted(
    [*data_dir.glob("*.csv"), *data_dir.glob("*.json")],
    key=lambda path: path.name.lower(),
)

if not dataset_files:
    raise FileNotFoundError("Nenhum arquivo CSV ou JSON foi encontrado na pasta 'data'.")

dataset_files

In [ ]:
def to_variable_name(file_path: Path) -> str:
    name = file_path.stem.lower()
    name = re.sub(r"\W+", "_", name)
    name = re.sub(r"^(\d)", r"df_\1", name)
    return name


dataframes = {}

for dataset_file in dataset_files:
    df_name = to_variable_name(dataset_file)
    df = read_dataset(dataset_file)
    dataframes[df_name] = df

list(dataframes.keys())

In [ ]:
for df_name, df in dataframes.items():
    print(f"{df_name}: {df.shape[0]} linhas x {df.shape[1]} colunas")

## 4) Simulation and Rewriting Utilities

In this section, we import utility functions and the provider enum to standardize calls (`Provider.GEMINI`, `Provider.OPENROUTER`, `Provider.LOCAL`).

In [ ]:
from utils.simulation_functions import rewrite_news_with_personality
from enums import Provider, Models

## 5) Simulation with Remote APIs

Next, we run rewriting simulations with remote providers and inspect `rewrite_status` and `rewrite_error`.

In [ ]:
DATASET = "newsdata_news"
TEST_MAX_ROWS = 20
PERSONALITY = "An enthusiastic science communicator who loves to make complex topics accessible and engaging for everyone. And aplies bias to the rewritten news, making it more sensational and clickbait-y, while still maintaining the core information of the original news article."

In [ ]:
gemini_rewritten_df = rewrite_news_with_personality(
    df=dataframes[DATASET],
    personality=PERSONALITY,
    provider=Provider.GEMINI,
    max_rows=TEST_MAX_ROWS,
    sleep_seconds=5.0,
    retry_attempts=2,
    text_column="description",
    model=Models.GEMINI31FlashLite,
)

print("Gemini status:")
print(gemini_rewritten_df["rewrite_status"].value_counts(dropna=False))

erros = gemini_rewritten_df[gemini_rewritten_df["rewrite_status"] == "error"]
if not erros.empty:
	print("\nErros em Gemini:")
	for idx, row in erros.iterrows():
		print(f"- linha {idx}: {row['rewrite_error']}")

In [ ]:
openrouter_nvidia_rewritten_df = rewrite_news_with_personality(
    df=dataframes[DATASET],
    personality=PERSONALITY,
    provider=Provider.OPENROUTER,
    model=Models.NEMOTRON3Super120B,
    max_rows=TEST_MAX_ROWS,
    sleep_seconds=5.0,
    retry_attempts=2,
    text_column="description",
)

print("OpenRouter (Nvidia Nemotron 3 Super free) status:")
print(openrouter_nvidia_rewritten_df["rewrite_status"].value_counts(dropna=False))

erros = openrouter_nvidia_rewritten_df[openrouter_nvidia_rewritten_df["rewrite_status"] == "error"]
if not erros.empty:
	print("\nErros em OpenRouter Nvidia:")
	for idx, row in erros.iterrows():
		print(f"- linha {idx}: {row['rewrite_error']}")

In [ ]:
openrouter_nvidia_rewritten_df = rewrite_news_with_personality(
    df=dataframes[DATASET],
    personality=PERSONALITY,
    provider=Provider.OPENROUTER,
    model=Models.STEP35FLASH,
    max_rows=TEST_MAX_ROWS,
    sleep_seconds=5.0,
    retry_attempts=2,
    text_column="description",
)

print("OpenRouter (Step 3.5 Flash) status:")
print(openrouter_nvidia_rewritten_df["rewrite_status"].value_counts(dropna=False))

erros = openrouter_nvidia_rewritten_df[openrouter_nvidia_rewritten_df["rewrite_status"] == "error"]
if not erros.empty:
	print("\nErros em OpenRouter Step 3.5 Flash:")
	for idx, row in erros.iterrows():
		print(f"- linha {idx}: {row['rewrite_error']}")

## 6) Local Model Test (Ollama)

In this section, we call a local OpenAI-compatible endpoint (`http://127.0.0.1:11434/v1`) to validate generation without external APIs.

In [ ]:
local_llama_rewritten_df = rewrite_news_with_personality(
    df=dataframes[DATASET],
    personality=PERSONALITY,
    text_column="description",
    provider=Provider.LOCAL,
    model=Models.LLAMA318B,
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    max_rows=TEST_MAX_ROWS,
    sleep_seconds=0.0,
    retry_attempts=2,
)

print("Local Llama status:")
print(local_llama_rewritten_df["rewrite_status"].value_counts(dropna=False))

local_llama_rewritten_df[[
    "title",
    "source_text_column",
    "target_language",
    "target_language_source",
    "rewritten_news",
    "rewrite_status",
    "rewrite_error",
]].head()

## 7) Export Simulation Outputs

In this step, we export final rewritten DataFrames to `../output/rewritten` for use in the audit notebooks.

In [ ]:
output_dir = Path("../output/rewritten")
output_dir.mkdir(parents=True, exist_ok=True)

def _safe_output_name(name: str) -> str:
    cleaned = re.sub(r"\W+", "_", name).strip("_").lower()
    return cleaned or "rewritten_dataset"

candidate_dfs: dict[str, pd.DataFrame] = {}
global_items = list(globals().items())

for var_name, var_value in global_items:
    if not isinstance(var_value, pd.DataFrame):
        continue

    # Exporta apenas resultados finais de reescrita, ex.: gemini_rewritten_df
    if not var_name.endswith("_rewritten_df"):
        continue

    required_columns = {"rewritten_news", "rewrite_status"}
    if not required_columns.issubset(set(var_value.columns)):
        continue

    candidate_dfs[var_name] = var_value

if not candidate_dfs:
    print("Nenhum DataFrame final de reescrita foi encontrado para exportacao.")
else:
    exported_files = []
    skipped_all_error = []

    for df_name in sorted(candidate_dfs.keys()):
        df_value = candidate_dfs[df_name]

        # Mantem apenas linhas bem-sucedidas (sem erro e com texto reescrito).
        export_df = df_value[
            (df_value["rewrite_status"] == "success")
            & df_value["rewritten_news"].notna()
            & (df_value["rewritten_news"].astype(str).str.strip() != "")
        ].copy()

        if export_df.empty:
            skipped_all_error.append(df_name)
            continue

        output_name = f"{_safe_output_name(df_name)}.csv"
        output_path = output_dir / output_name
        export_df.to_csv(output_path, index=False, encoding="utf-8-sig")
        exported_files.append(output_path)

    print(f"Total de DataFrames exportados: {len(exported_files)}")
    for file_path in exported_files:
        print(f"- {file_path}")

    if skipped_all_error:
        print("\nDataFrames nao exportados (sem linhas validas):")
        for df_name in skipped_all_error:
            print(f"- {df_name}")